In [1]:
print(0)

0


In [2]:
#!/usr/bin/env python3
"""
Skin-tone stratified model performance tables.

  FP skin tones 1-6  ×  BioClinBERT P / R / F  =  18 tables
  Each table: rows = models, columns = 15 features + avg_score
  OSD and RPD are combined (patients/visits matched via FP annotations).

Output CSVs: OUTPUT_DIR/fp{tone}_{p|r|f}_bioclinbert.csv
"""

import os
import numpy as np
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────────────────────
FP_OSD_CSV = "/home/pratyushprasad/ACL/FP_ANNOTATIONS/FITZPATRICK_ANNOTATIONS_OPEN_SOURCE.csv"
FP_RPD_CSV = "/home/pratyushprasad/ACL/FP_ANNOTATIONS/FITZPATRICK_ANNOTATIONS_REAL_PATIENT.csv"
SCORES_OSD = "/home/pratyushprasad/ACL/SCORES/OSD"
SCORES_RPD = "/home/pratyushprasad/ACL/SCORES/RPD"
OUTPUT_DIR = "/home/pratyushprasad/ACL/SCORES/skin_tone_tables"

BIOCLIN_FILES = {
    "p_bioclinbert.csv": "Precision",
    "r_bioclinbert.csv": "Recall",
    "f_bioclinbert.csv": "F1",
}
VALID_STATUSES = {"agreed", "adjudicated"}

ALL_COLS = [
    "type_of_lesion", "site", "count", "arrangement", "size",
    "color_pattern", "border", "surface_changes",
    "presence_of_exudate_or_discharge", "surrounding_skin_changes",
    "secondary_changes", "pattern_or_shape", "differential_diagnosis",
    "additional_notes", "overall_description",
]

FP_NAMES = {
    1: "FP1 — Pale white",
    2: "FP2 — White",
    3: "FP3 — Light brown/olive",
    4: "FP4 — Moderate brown",
    5: "FP5 — Dark brown",
    6: "FP6 — Deeply pigmented",
}


# ── Load FP annotation lookup ─────────────────────────────────────────────────

def load_fp_lookup():
    """
    Returns two dicts:
      osd_fp : { patient_id_str → fp_int }
      rpd_fp : { (pid,cid,vid)  → fp_int }
    Only rows with STATUS in VALID_STATUSES and FINAL in 1-6 are included.
    """
    def to_fp(v):
        try:
            iv = int(float(str(v).strip()))
            return iv if 1 <= iv <= 6 else None
        except (ValueError, TypeError):
            return None

    osd_fp = {}
    df = pd.read_csv(FP_OSD_CSV, dtype=str)
    for _, row in df.iterrows():
        if str(row.get("STATUS", "")).strip().lower() not in VALID_STATUSES:
            continue
        fp = to_fp(row.get("FINAL"))
        if fp:
            osd_fp[str(row["patient_id"]).strip()] = fp

    rpd_fp = {}
    df = pd.read_csv(FP_RPD_CSV, dtype=str)
    for _, row in df.iterrows():
        if str(row.get("STATUS", "")).strip().lower() not in VALID_STATUSES:
            continue
        fp = to_fp(row.get("FINAL"))
        if fp:
            key = (
                str(row["patient_id"]).strip(),
                str(row["complaint_id"]).strip(),
                str(row["visit_id"]).strip(),
            )
            rpd_fp[key] = fp

    print(f"  FP lookup — OSD: {len(osd_fp)} patients | RPD: {len(rpd_fp)} visits")
    return osd_fp, rpd_fp


# ── Load scores and tag with FP class ─────────────────────────────────────────

def load_scores(scores_root, source, osd_fp, rpd_fp):
    """
    Walk scores_root/<model>/<bioclin_file>.csv.
    Returns list of dicts: {model, metric, fp_class, col1..col15}
    """
    records = []

    for model_name in sorted(os.listdir(scores_root)):
        model_path = os.path.join(scores_root, model_name)
        if not os.path.isdir(model_path):
            continue

        for fname, metric_label in BIOCLIN_FILES.items():
            fpath = os.path.join(model_path, fname)
            if not os.path.exists(fpath):
                continue

            df = pd.read_csv(fpath, dtype=str)

            for _, row in df.iterrows():
                # Resolve FP class from lookup
                if source == "OSD":
                    pid = str(row.get("pid", "")).strip()
                    fp  = osd_fp.get(pid)
                else:
                    key = (
                        str(row.get("pid", "")).strip(),
                        str(row.get("cid", "")).strip(),
                        str(row.get("vid", "")).strip(),
                    )
                    fp = rpd_fp.get(key)

                if fp is None:
                    continue  # no FP annotation for this patient

                rec = {"model": model_name, "metric": fname, "fp_class": fp}
                for col in ALL_COLS:
                    try:
                        rec[col] = float(row.get(col, float("nan")))
                    except (ValueError, TypeError):
                        rec[col] = float("nan")

                records.append(rec)

        print(f"    {source}  {model_name}: loaded")

    return records


# ── Build and save tables ──────────────────────────────────────────────────────

def build_tables(all_records):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    df = pd.DataFrame(all_records)

    if df.empty:
        print("  [WARN] No records found — check paths and FP annotations.")
        return

    print(f"\n  Total matched records: {len(df)}")
    print(f"  FP class distribution:\n{df['fp_class'].value_counts().sort_index().to_string()}")

    for fp in range(1, 7):
        for fname, metric_label in BIOCLIN_FILES.items():
            sub = df[(df["fp_class"] == fp) & (df["metric"] == fname)]

            if sub.empty:
                print(f"\n  [SKIP] FP={fp} | {metric_label} — no data")
                continue

            # Mean per model across all matched patients
            table = (
                sub.groupby("model")[ALL_COLS]
                .mean(numeric_only=True)
                .dropna(how="all")
            )
            table["avg_score"] = table[ALL_COLS].mean(axis=1, skipna=True)
            table = table.sort_values("avg_score", ascending=False)
            table = table.round(4)

            # ── Print ──────────────────────────────────────────────────────────
            sep    = "─" * (20 + 10 * (len(ALL_COLS) + 1))
            header = f"  FP={fp} ({FP_NAMES[fp]})  |  BioClinBERT {metric_label}  (n_patients={len(sub['model'].eq(sub['model']).index.unique())})"
            print(f"\n{'='*len(sep)}")
            print(header)
            print(f"{'='*len(sep)}")
            print(table.to_string())
            print()

            # ── Save ──────────────────────────────────────────────────────────
            prefix   = fname.replace("_bioclinbert.csv", "")
            out_path = os.path.join(OUTPUT_DIR, f"fp{fp}_{prefix}_bioclinbert.csv")
            table.reset_index().to_csv(out_path, index=False)
            print(f"  → saved: {out_path}")


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print("\n" + "=" * 70)
    print("  SKIN-TONE STRATIFIED SCORES  (BioClinBERT P / R / F)")
    print("=" * 70)

    osd_fp, rpd_fp = load_fp_lookup()

    print(f"\n  Loading OSD scores...")
    osd_records = load_scores(SCORES_OSD, "OSD", osd_fp, rpd_fp)

    print(f"\n  Loading RPD scores...")
    rpd_records = load_scores(SCORES_RPD, "RPD", osd_fp, rpd_fp)

    all_records = osd_records + rpd_records
    print(f"\n  Combined: {len(all_records)} scored rows with FP annotation")

    build_tables(all_records)

    print("\n" + "=" * 70)
    print(f"  DONE — 18 CSVs saved to {OUTPUT_DIR}")
    print("=" * 70 + "\n")


if __name__ == "__main__":
    main()


  SKIN-TONE STRATIFIED SCORES  (BioClinBERT P / R / F)
  FP lookup — OSD: 247 patients | RPD: 532 visits

  Loading OSD scores...
    OSD  dermatollama: loaded
    OSD  gemini: loaded
    OSD  lingshu_32b: loaded
    OSD  lingshu_7b: loaded
    OSD  llava_med: loaded
    OSD  med_gemma_27b: loaded
    OSD  med_gemma_4b: loaded
    OSD  med_r1_dermo: loaded
    OSD  ours_27b_06__00: loaded
    OSD  ours_27b_07__00: loaded
    OSD  ours_27b_08__00: loaded
    OSD  ours_27b_09__00: loaded
    OSD  ours_27b_10__00: loaded
    OSD  ours_4b_06__00: loaded
    OSD  ours_4b_07__00: loaded
    OSD  ours_4b_08__00: loaded
    OSD  ours_4b_09__00: loaded
    OSD  ours_4b_10__00: loaded
    OSD  qwen_72b: loaded
    OSD  skinvl: loaded

  Loading RPD scores...
    RPD  dermatollama: loaded
    RPD  gemini: loaded
    RPD  lingshu_32b: loaded
    RPD  lingshu_7b: loaded
    RPD  llava_med: loaded
    RPD  med_gemma_27b: loaded
    RPD  med_gemma_4b: loaded
    RPD  med_r1_dermo: loaded
    RPD  ou